In [0]:
from pyspark.sql import functions as F

In [0]:
BRONZE_TABLE = "traffic_project.bronze.vehicles"
SILVER_TABLE = "traffic_project.silver.track_events"
QUARANTINE_TABLE = "traffic_project.silver.quarantine_events"

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

In [0]:
valid_condition = F.coalesce(
    (
        F.col("event_id").isNotNull()
        & F.col("batch_id").isNotNull()
        & F.col("camera_id").isNotNull()
        & F.col("track_id").isNotNull()
        & F.col("track_status").isin("active", "lost")
        & F.col("event_time").isNotNull()
        & F.col("vehicle_type").isin(
            "car", "motorcycle", "truck", "bus"
        )
        & F.col("detection_confidence").between(0.0, 1.0)
        & F.col("roi_zone").isNotNull()
    ),
    F.lit(False)
)


In [0]:
silver_df = (
    bronze_df
    .filter(valid_condition)
    .withColumn(
        "quality_flag",
        F.when(F.col("track_status") == "lost", "track_lost")
         .otherwise("ok")
    )
    .withColumn("validated_at", F.current_timestamp())
    .dropDuplicates(["event_id"])
)

In [0]:
quarantine_df = (
    bronze_df
    .filter(~valid_condition)
    .withColumn(
        "reason",
        F.when(F.col("track_id").isNull(), "missing_track_id")
         .when(F.col("event_time").isNull(), "missing_event_time")
         .when(F.col("detection_confidence").isNull(), "missing_confidence")
         .otherwise("invalid_required_field_or_value")
    )
    .withColumn("quarantined_at", F.current_timestamp())
)


In [0]:
silver_df.createOrReplaceTempView("silver_updates")
quarantine_df.createOrReplaceTempView("quarantine_updates")

In [0]:

spark.sql("""
CREATE TABLE IF NOT EXISTS traffic_project.silver.track_events
USING DELTA
AS
SELECT *
FROM silver_updates
WHERE 1 = 0
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS traffic_project.silver.quarantine_events
USING DELTA
AS
SELECT *
FROM quarantine_updates
WHERE 1 = 0
""")

In [0]:
spark.sql("""
MERGE INTO traffic_project.silver.track_events AS target
USING silver_updates AS source
ON target.event_id = source.event_id

WHEN NOT MATCHED THEN
  INSERT *
""")

In [0]:
spark.sql("""
MERGE INTO traffic_project.silver.quarantine_events AS target
USING quarantine_updates AS source
ON target.event_id = source.event_id
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
bronze_count = spark.table(
    "traffic_project.bronze.vehicles"
).count()

silver_count = spark.table(
    "traffic_project.silver.track_events"
).count()

quarantine_count = spark.table(
    "traffic_project.silver.quarantine_events"
).count()

print("Bronze:", bronze_count)
print("Silver:", silver_count)
print("Quarantine:", quarantine_count)

assert bronze_count == silver_count + quarantine_count